# Multiagent Gridhunt

Student Name: Phan Lê Ngọc Như Ý (MSSV: 3124720079) - Nhóm: Phan Lê Ngọc Như Ý, Phạm Thái Duy Khang, Bùi Anh Đức, Bùi Quyền Kiệt

I have used the following AI tools: ChatGPT, GitHub Copilot

I understand that my submission needs to be my own work: PLNNY


## Learning Outcomes

* Apply core AI concepts by implementing the agent function for a simple and model-based reflex agents that respond to environmental percepts.
* Practice how the environment and the agent function interact.
* Analyze agent performance through controlled experiments across different environment configurations.

## Instructions

Total Points: 10

Complete this notebook. Use the provided notebook cells and insert additional code and markdown cells as needed. Submit the completely rendered notebook.
### AI Use

Here are some guidelines that will make it easier for you:

* __Don't:__ Rely on AI auto completion. You will waste a lot of time trying to figure out how the suggested code relates to what we do in class. Turn off AI code completion (e.g., Copilot) in your IDE.
* __Don't:__ Do not submit code/text that you do not understand or have not checked to make sure that it is complete and correct.
* __Do:__ Use AI for debugging and letting it explain code and concepts from class.

## Introduction

![GridHunt Image](gridhunt.png)

Gridhunt is a small educational game that helps you practice implementing simple agent functions. It is a modified Python reimplementation of Gridhunt2. 

The objective of gridhunt is to implement an intelligent agent, the hunter, who can catch a monster faster than all other hunters. Gridhunt uses a $n \times n$ arena consisting of a grid of tiles. Gridhunt is a turn-based game, and several hunter agents and the monster agent move around in the arena. A hunter catches the monster if she/he moves on the same square the monster currently occupies. The hunters compete fow who catches the monster first. If the monster survives the maximum number of steps for the game, then the monster wins.

## PEAS Description of Gridhunt

__Performance Measure:__ The performance is measured as the number of steps the hunter uses to catch the monster. Catching the monster means to be on the same square.

__Environment:__ An arena with $n \times n$ squares. At the beginning the monster and the hunters are randomly placed in the arena environment. The hunters and the monster
    can move around, but cannot leave the arena. Since there are three agents involved, this is a competitive multi-agent environment.

__Actuators:__ The agent can move to an adjacent square using actions "north", "east", "west", and "south", or teleport which will move the agent 
    to a random square in the arena.

__Sensors:__ The agent can always see its location in the arena and the location of the monster.  

## The Arena Environment

The environment manages the location of the agents (hunter and monster). It initially places them in a random location and then provides them in every step with percepts (the location of the hunter and the monster) and asks them for their action. 

__A Note on positions:__
The arena is implemented as an array with row and column indices representing the position.
Positions are stored in a numpy array where `pos[0]` is the row index and `pos[1]` is the column index in the arena. North means up in this array, that is the row index gets smaller when going north. Remember indices in Python start with 0 and go to `n-1`.

In [2]:
import numpy as np
from IPython.display import clear_output
from time import sleep

def arena_environment(hunter_1_agent_function, hunter_2_agent_function, monster_agent_function, n = 30, max_steps = 100, visualize = False, animation = False):
    """
    Simulate an arena where two hunter agent tries to catch a monster agent.

    Parameters:
    hunter_1_agent_function and hunter_2_agent_function (function): A function that takes the current positions of the hunter and monster
                                      and returns the next move for the hunter ('north', 'east', 'west', 'south', 'teleport').
    monster_agent_function (function): A function that takes the current positions of the hunter and monster
                                       and returns the next move for the monster (or 'stay' to remain in place).
    n (int): The size of the arena (n x n grid).
    max_steps (int): The maximum number of steps to simulate.
    visualize (bool): Whether to visualize the arena.
    animation (bool): Whether to animate the visualization with a delay.

    Returns:
    int: The number of steps taken for the hunter to catch the monster or np.nan (not a number) if not caught.
    """
    
    # Initialize positions
    monster_pos = np.random.randint(0, n-1, size=2)
    hunter_1_pos = np.random.randint(0, n-1, size=2)
    while np.array_equal(hunter_1_pos, monster_pos):
        hunter_1_pos = np.random.randint(0, n-1, size=2)
    hunter_2_pos = np.random.randint(0, n-1, size=2)
    while np.array_equal(hunter_2_pos, monster_pos):
        hunter_2_pos = np.random.randint(0, n-1, size=2)    
    
    def move(action, position):
        """calculate new position for the agent."""
        if action == 'north':
            position[0] -= 1
        elif action == 'south':
            position[0] += 1
        elif action == 'west':
            position[1] -= 1
        elif action == 'east':
            position[1] += 1
        else:
            raise ValueError("Invalid action.")
        
        # Ensure position stays within bounds
        position = np.clip(position, 0, n-1)
        return position

    def print_arena():

        if animation:
            sleep(1)
            clear_output(wait=True)
        print(f"Step {step}:")
        print(f"Hunter 1 is at '{hunter_1_pos}'; Hunter 2 is at '{hunter_2_pos}'; Monster is at '{monster_pos}'\n")
        arena = np.full((n, n), '.', dtype=str)
        arena[monster_pos[0], monster_pos[1]] = 'M'
        arena[hunter_1_pos[0], hunter_1_pos[1]] = '1'
        arena[hunter_2_pos[0], hunter_2_pos[1]] = '2'
        print("\n".join("".join(row) for row in arena))

    for step in range(max_steps):
        if visualize:
            print_arena()
        
        # Check if hunter has caught the monster
        if np.array_equal(hunter_1_pos, monster_pos):
            if visualize:
                print(f"Hunter 1 caught the monster in {step} steps!")
            return ("1", step)
        
        if np.array_equal(hunter_2_pos, monster_pos):
            if visualize:
                print(f"Hunter 2 caught the monster in {step} steps!")
            return ("2", step)

        # Get next move from monster agent function
        monster_action = monster_agent_function(monster_pos, hunter_1_pos, hunter_2_pos)
        if monster_action != 'stay':
            monster_pos = move(monster_action, monster_pos)

        # Get next move from hunter agent function
        hunter_1_action = hunter_1_agent_function(hunter_1_pos, monster_pos)
        if hunter_1_action == 'teleport':
            hunter_1_pos = np.random.randint(0, n-1, size=2)
        else:
            hunter_1_pos = move(hunter_1_action, hunter_1_pos)

        hunter_2_action = hunter_2_agent_function(hunter_2_pos, monster_pos)
        if hunter_2_action == 'teleport':
            hunter_2_pos = np.random.randint(0, n-1, size=2)
        else:
            hunter_2_pos = move(hunter_2_action, hunter_2_pos)
    
        # print the agents' actions
        if visualize:
            print(f"Hunter 1 chose action '{hunter_1_action}'")
            print(f"Hunter 2 chose action '{hunter_2_action}'")
            print(f"Monster chose action '{monster_action}'")
            print("\n" + "="*n + "\n")

    # the hunter failed to catch the monster within max_steps
    if visualize:
        print(f"Monster won by surviving {max_steps} steps.")
    
    return ("M", np.nan)

I implement the monster here as a simple agent that moves around randomly, but mostly stays in its place. You can use AI to get a detailed explanation of how the following code works.

In [3]:
monster_actions = ["north", "east", "west", "south", "stay"]

def monster_agent_function_simple(monster_pos, hunter_1_pos, hunter_2_pos):
    return np.random.choice(monster_actions, p=[0.125, 0.125, 0.125, 0.125, 0.5])

# The Hunter Agent

Your job is to implement a simple-reflex hunter agent that can catch the monster. Remember, your agent is implemented as an agent function that get percepts and needs to return a valid action. The actions are: 

In [4]:
actions = ["north", "east", "west", "south", "teleport"]

## A Simple Example Implementation

Here is a very simple hunter that just runs around randomly and hopes that it bumps into the monster. 

In [5]:
def simple_randomized_hunter_agent_function(hunter_location, monster_location):
    return np.random.choice(actions)

Ask the agent for an action.

In [11]:
simple_randomized_hunter_agent_function([0,0], [5,5])

np.str_('south')

## Experimenting With the Agent

We can place the monster and the hunter into the environment and run a simulation by calling the environment function.

In [7]:
hunter_1 = simple_randomized_hunter_agent_function
hunter_2 = simple_randomized_hunter_agent_function

arena_environment(hunter_1, 
                  hunter_2, 
                  monster_agent_function_simple, 
                  n=5, max_steps=5, visualize=True, animation=False)

Step 0:
Hunter 1 is at '[2 1]'; Hunter 2 is at '[1 3]'; Monster is at '[2 0]'

.....
...2.
M1...
.....
.....
Hunter 1 chose action 'teleport'
Hunter 2 chose action 'east'
Monster chose action 'stay'

=====

Step 1:
Hunter 1 is at '[3 0]'; Hunter 2 is at '[1 4]'; Monster is at '[2 0]'

.....
....2
M....
1....
.....
Hunter 1 chose action 'south'
Hunter 2 chose action 'teleport'
Monster chose action 'stay'

=====

Step 2:
Hunter 1 is at '[4 0]'; Hunter 2 is at '[3 0]'; Monster is at '[2 0]'

.....
.....
M....
2....
1....
Hunter 1 chose action 'west'
Hunter 2 chose action 'south'
Monster chose action 'west'

=====

Step 3:
Hunter 1 is at '[4 0]'; Hunter 2 is at '[4 0]'; Monster is at '[2 0]'

.....
.....
M....
.....
2....
Hunter 1 chose action 'north'
Hunter 2 chose action 'west'
Monster chose action 'stay'

=====

Step 4:
Hunter 1 is at '[3 0]'; Hunter 2 is at '[4 0]'; Monster is at '[2 0]'

.....
.....
M....
1....
2....
Hunter 1 chose action 'south'
Hunter 2 chose action 'east'
Monster c

('M', nan)

Well, this was one run, maybe it was just good or bad luck this time!

Let's run an experiment with 100 runs in a $20 \times 20$ arena.  

In [12]:
hunter_1 = simple_randomized_hunter_agent_function
hunter_2 = simple_randomized_hunter_agent_function

steps = [arena_environment(hunter_1, 
                           hunter_2, 
                           monster_agent_function_simple, 
                           n=20, max_steps=100, visualize=False) for _ in range(100)] 
print(steps)

[('M', nan), ('M', nan), ('M', nan), ('M', nan), ('1', 70), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('1', 14), ('M', nan), ('2', 19), ('M', nan), ('M', nan), ('1', 41), ('M', nan), ('M', nan), ('1', 47), ('M', nan), ('M', nan), ('2', 59), ('1', 4), ('M', nan), ('1', 6), ('2', 43), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('2', 88), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('1', 23), ('M', nan), ('M', nan), ('1', 54), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('1', 41), ('M', nan), ('M', nan), ('1', 51), ('M', nan), ('2', 54), ('M', nan), ('M', nan), ('1', 19), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('M', nan), ('1', 5), ('1', 9), ('M', nan), ('2', 53), ('M', nan), ('1', 44), ('M', nan), ('M', nan), ('M', nan), ('1', 61), ('M', nan), ('M', nan), ('1', 82), ('M', nan), ('2', 24), ('1', 98), ('M', nan), ('2', 32), ('1', 26), ('1', 37)

We just got a list with who one and the number of steps to catch the monster for 100 simulation runs.
Let's analysis the results by counting who won how often.

In [13]:
steps = np.array(steps)

print (f"Hunter 1 won: {np.sum(steps[:,0] == "1")}")
print (f"Hunter 2 won: {np.sum(steps[:,0] == "2")}")
print (f"Monster 1 won: {np.sum(steps[:,0] == "M")}")       


Hunter 1 won: 21
Hunter 2 won: 10
Monster 1 won: 69


The random agent in not a very good hunter. You need to implement a better agent function.

# Your Agent Implementation [4 points]

Write a new hunter agent function that chases the monster. Copy the simple randomized hunter function from above and modify it using rules based on the monster's and the hunter's location. Let your hunter compete with the random agent. You can also come up with several different agents and let them compete against each other.

In [14]:
def greedy_reflex_hunter_agent_function(hunter_location, monster_location):
    """A simple-reflex hunter agent.

    The agent only uses the CURRENT percept (its own position and the
    monster's position) -- it has no memory of the past. On every call it
    picks the single move ('north', 'east', 'west', 'south') that reduces
    the Manhattan (grid) distance to the monster the most.

    Strategy:
    1. Compute how far the monster is in the row direction (dr) and in the
       column direction (dc).
    2. Close the gap along whichever axis has the larger absolute
       difference first (this is the axis where a move buys the most
       distance reduction).
    3. If the hunter and monster are already aligned on that axis
       (difference == 0), close the gap on the other axis instead.
    4. Teleportation is not used here: since 'teleport' sends the agent to
       a uniformly random square, it is very unlikely to move the hunter
       closer to the monster on purpose, so a directed step is almost
       always the better choice for a chasing strategy.
    """
    dr = monster_location[0] - hunter_location[0]  # row difference (north/south)
    dc = monster_location[1] - hunter_location[1]  # column difference (west/east)

    # Move along whichever axis currently has the bigger gap
    if abs(dr) >= abs(dc) and dr != 0:
        return "south" if dr > 0 else "north"
    elif dc != 0:
        return "east" if dc > 0 else "west"
    elif dr != 0:
        return "south" if dr > 0 else "north"
    else:
        # Same square (shouldn't normally happen -- the environment checks
        # for a catch before asking for the next move) -- pick anything.
        return np.random.choice(actions)


# Your Experiments [2 points]

Copy the simulation code from above and run experiments with your agent. 
Experiment with larger arenas of at least size $30 \times 30$.

In [15]:
# Let my greedy hunter compete against the simple random hunter
hunter_1 = greedy_reflex_hunter_agent_function
hunter_2 = simple_randomized_hunter_agent_function

results_30 = [arena_environment(hunter_1,
                                 hunter_2,
                                 monster_agent_function_simple,
                                 n=30, max_steps=100, visualize=False)
              for _ in range(100)]

results_30 = np.array(results_30)
print("Arena 30x30, greedy hunter (1) vs. random hunter (2):")
print(f"Hunter 1 (greedy) won: {np.sum(results_30[:, 0] == '1')}")
print(f"Hunter 2 (random) won: {np.sum(results_30[:, 0] == '2')}")
print(f"Monster won:           {np.sum(results_30[:, 0] == 'M')}")

# Now let two copies of the greedy hunter compete against each other,
# and try a bigger arena to see how arena size affects performance.
hunter_1 = greedy_reflex_hunter_agent_function
hunter_2 = greedy_reflex_hunter_agent_function

for size in [30, 50, 80]:
    results = [arena_environment(hunter_1,
                                  hunter_2,
                                  monster_agent_function_simple,
                                  n=size, max_steps=200, visualize=False)
               for _ in range(100)]
    results = np.array(results)
    steps_taken = results[results[:, 0] != "M", 1].astype(float)

    print(f"--- Arena {size}x{size} ---")
    print(f"Hunter 1 won: {np.sum(results[:, 0] == '1')}")
    print(f"Hunter 2 won: {np.sum(results[:, 0] == '2')}")
    print(f"Monster won:  {np.sum(results[:, 0] == 'M')}")
    if len(steps_taken) > 0:
        print(f"Average steps to catch (when caught): {steps_taken.mean():.1f}")
    print()


Arena 30x30, greedy hunter (1) vs. random hunter (2):
Hunter 1 (greedy) won: 97
Hunter 2 (random) won: 3
Monster won:           0
--- Arena 30x30 ---
Hunter 1 won: 56
Hunter 2 won: 44
Monster won:  0
Average steps to catch (when caught): 16.2

--- Arena 50x50 ---
Hunter 1 won: 49
Hunter 2 won: 51
Monster won:  0
Average steps to catch (when caught): 24.9

--- Arena 80x80 ---
Hunter 1 won: 47
Hunter 2 won: 53
Monster won:  0
Average steps to catch (when caught): 41.1



## Your Conclusion [4 points] 

Discuss the following:

* What is your hunter's final strategy to choose actions. Why does it work well?
* Do you use teleportation. Why and in what situation? Why not?
* How does the arena size affects your hunter agent's performance?

> **Chiến lược của thợ săn:** Tác tử thợ săn của tôi là một **tác tử phản xạ đơn giản (simple-reflex agent)**: tại mỗi lượt, tác tử chỉ nhìn vào vị trí hiện tại của chính nó và vị trí của quái vật (tri giác vừa nhận được) và chọn duy nhất một bước đi giúp giảm khoảng cách lưới Manhattan giữa hai bên nhiều nhất. Cụ thể, tác tử so sánh độ lệch theo hàng (`row gap`) và độ lệch theo cột (`column gap`) tới quái vật, rồi ưu tiên bước theo trục có độ lệch lớn hơn; khi hai bên đã thẳng hàng trên một trục, tác tử tiếp tục rút ngắn khoảng cách trên trục còn lại. Nhờ việc thu hẹp khoảng cách một cách tham lam ở mỗi bước, thợ săn hội tụ tới ô của quái vật trong khoảng `|dr| + |dc|` bước bất cứ khi nào quái vật đứng yên hoặc di chuyển chậm — đây là hiệu quả tiệm cận mức tối ưu đối với một tác tử không có bộ nhớ và không lập kế hoạch dài hạn. Chiến lược này vượt trội hoàn toàn so với thợ săn ngẫu nhiên thuần túy vốn không hề có ý niệm về hướng và chỉ thực hiện bước đi ngẫu nhiên không định hướng (unbiased random walk).
>
> **Về hành động dịch chuyển tức thời (`teleport`):** Tác tử của tôi không bao giờ sử dụng "teleport". Hành động teleport đưa tác tử tới một ô *ngẫu nhiên đồng đều* trong đấu trường, hoàn toàn độc lập với vị trí của quái vật. Do đó, về mặt trung bình, teleport kém hiệu quả hơn rất nhiều so với một bước đi có định hướng được đảm bảo sẽ thu hẹp khoảng cách tới quái vật. Teleport chỉ thực sự có ý nghĩa như một giải pháp cứu cánh cuối cùng nếu thợ săn bị kẹt vào một góc hoặc đường cụt trong môi trường có vật cản; tuy nhiên trong môi trường đấu trường mở này, thợ săn luôn có ít nhất một hướng di chuyển hữu ích. Vì vậy, việc đánh cược với rủi ro bị dịch chuyển văng ra xa quái vật là không đáng giá.
>
> **Ảnh hưởng của kích thước đấu trường:** Khi đấu trường mở rộng (ví dụ từ 30×30 lên 80×80), số bước trung bình cần thiết để bắt được quái vật tăng lên rõ rệt. Nguyên nhân là vì khoảng cách Manhattan ban đầu giữa thợ săn và quái vật tỷ lệ thuận với chiều dài cạnh $n$ của đấu trường (khoảng cách kỳ vọng tăng tuyến tính theo $n$). Quái vật cũng có không gian rộng lớn hơn để lẩn trốn trước khi thợ săn kịp áp sát. Với một ngân sách bước cố định (`max_steps`), đấu trường lớn làm tăng khả năng quái vật sống sót đến khi hết giờ và "chiến thắng" nhờ việc kéo dài thời gian lẩn tránh, mặc dù bản thân chiến lược săn mồi của thợ săn không hề kém đi — nó chỉ phải đối mặt với một quãng đường truy đuổi kỳ vọng dài hơn nhiều.


The monster is also an agent. Describe how the monster's agent function could be changed so it gets better at avoiding the hunter.

> Hàm tác tử quái vật vốn đã nhận được vị trí của chính nó cùng vị trí của cả hai thợ săn tại mỗi lượt (`monster_pos, hunter_1_pos, hunter_2_pos`), do đó nó hoàn toàn có đủ thông tin để né tránh một cách chủ động thay vì đi ngẫu nhiên:
>
> 1. **Né tránh dựa trên lực đẩy kết hợp (Repulsion Vector):** Tại mỗi bước, quái vật có thể tính khoảng cách Manhattan tới cả hai thợ săn $d_1$ và $d_2$. Quái vật sẽ đánh giá 4 hướng di chuyển khả dĩ và chọn hướng làm tối đa hóa hàm khoảng cách kết hợp (ví dụ: tối đa hóa $\min(d_1, d_2)$ hoặc $d_1 + d_2$). Điều này đảm bảo quái vật luôn bước ra xa người thợ săn đang ở cự ly gần nhất và tránh bị kẹp vào giữa hai thợ săn.
>
> 2. **Dịch chuyển tức thời thoát hiểm có điều kiện:** Nếu cả hai thợ săn đều đã áp sát trong phạm vi nguy hiểm (ví dụ: $\min(d_1, d_2) \le 2$), quái vật có thể kích hoạt hành động `teleport` để nhảy tới một vị trí ngẫu nhiên khác trên bàn cờ. Cơ chế này sẽ phá vỡ thế bao vây của hai thợ săn và buộc họ phải bắt đầu lại quá trình truy đuổi từ khoảng cách xa.


# More Work (optional)

* Improve the monster agent so it tries to run away from the hunters. Don;t make the monster too fast or the hunters will never be able to catch it.
* How could the two hunter agents work together? You need to think about information sharing. This can be implemented by the two agent functions sharing a single Python class or the environment could be expanded by an action "shout" that lets the other agent perceive a message.